# Stage 2 — Instruction Fine-Tuning

Instruction tuning of the Stage 1 domain-adapted `Qwen/Qwen2.5-0.5B` on California homeowners-insurance instruction/response pairs using [Unsloth](https://github.com/unslothai/unsloth), LoRA/QLoRA, on a free Colab T4.

Spec: `specs/002-unsloth-ca-homeowners-finetune.md` §6-7 (Stage 2).

**Run this notebook on Google Colab with a T4 GPU runtime** (Runtime → Change runtime type → T4 GPU). It has not been executed locally — there is no CUDA GPU in the local dev environment for this repo.

Before running: upload/clone this repo into the Colab session (`!git clone <repo-url>` then `%cd <repo-name>`), and run Stage 1 (`notebooks/1-non_instruction_finetuning.ipynb`) first so `models/stage1-merged/` exists — this notebook continues from that checkpoint. If it's missing, this notebook falls back to the raw base model and prints a warning.

## 0. Install dependencies

In [ ]:
%%capture
import importlib.util

if importlib.util.find_spec("unsloth") is None:
    !pip install unsloth
    !pip install --no-deps "trl>=0.9.0" "peft>=0.11.0" "accelerate>=0.30.0" "bitsandbytes>=0.43.0"

In [ ]:
import sys
from pathlib import Path

# Assumes this notebook is run from notebooks/ inside the cloned repo, so the
# repo root (one level up) contains the src/ package and data/ directory.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_PATH = REPO_ROOT / "data" / "2-instruction_dataset.jsonl"
STAGE1_MODEL_DIR = REPO_ROOT / "models" / "stage1-merged"
ADAPTER_OUT_DIR = REPO_ROOT / "models" / "stage2-instruction-adapter"
MERGED_OUT_DIR = REPO_ROOT / "models" / "stage2-merged"
print("Repo root:", REPO_ROOT)
print("Data path exists:", DATA_PATH.exists())
print("Stage 1 merged model exists:", STAGE1_MODEL_DIR.exists())

## 1. Load the instruction dataset

In [ ]:
from src.data_utils import load_jsonl

pairs = load_jsonl(DATA_PATH)
print(f"Loaded {len(pairs)} instruction/response pairs")
print(pairs[0])

## 2. Load the Stage 1 model (continuing from Stage 1)

Loads the Stage 1 **merged** checkpoint (base weights with the Stage 1 domain adaptation baked in) as this stage's starting "base" model, then attaches a fresh set of Stage 2 LoRA adapters below (spec §6.4) — rather than resuming training on top of the Stage 1 PEFT wrapper directly.

In [ ]:
# If models/stage1-merged/ isn't already present in this session, upload the
# stage1-merged.zip downloaded at the end of Stage 1 and extract it here.
if not STAGE1_MODEL_DIR.exists():
    from google.colab import files

    print("Upload stage1-merged.zip from Stage 1:")
    uploaded = files.upload()
    STAGE1_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    for fname in uploaded:
        !unzip -q "{fname}" -d "{STAGE1_MODEL_DIR}"
    print("Extracted to:", STAGE1_MODEL_DIR)

In [ ]:
from src.model_utils import BASE_MODEL_NAME, MAX_SEQ_LENGTH, load_base_model

stage1_source = STAGE1_MODEL_DIR if STAGE1_MODEL_DIR.exists() else BASE_MODEL_NAME
if stage1_source == BASE_MODEL_NAME:
    print("Stage 1 merged model not found locally — falling back to the raw base model.")

model, tokenizer = load_base_model(
    model_name=str(stage1_source),
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
print("Loaded:", stage1_source, "| eos_token:", tokenizer.eos_token)

## 3. Format the instruction dataset with the chat template

In [ ]:
from src.data_utils import build_chat_dataset

dataset = build_chat_dataset(pairs, tokenizer)
print(dataset)
print(dataset[0]["text"][:500])

## 4. Apply fresh LoRA / QLoRA adapters for Stage 2

In [ ]:
from src.model_utils import add_lora_adapters

model = add_lora_adapters(model)  # uses DEFAULT_LORA_CONFIG: r=16, alpha=16, dropout=0.05

## 5. Train on the instruction dataset

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=str(REPO_ROOT / "outputs" / "stage2"),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,  # effective batch size 8, per spec §6.3
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="adamw_8bit",
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,  # each example is already one full chat turn, not raw text to pack
    args=training_args,
)

train_result = trainer.train()
print(train_result.metrics)

## 6. Save the Stage 2 adapter and merged model

Colab local disk is not persistent across sessions — copy these directories to Google Drive or push them to a private Hugging Face model repo after this cell (see spec §6.4).

In [ ]:
ADAPTER_OUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ADAPTER_OUT_DIR))
tokenizer.save_pretrained(str(ADAPTER_OUT_DIR))
print("Saved adapter to:", ADAPTER_OUT_DIR)

In [ ]:
# Persist Stage 2 outputs by downloading them as a zip — this avoids Colab's
# occasionally-flaky Drive-mount OAuth flow ("credential propagation was
# unsuccessful"). Upload the resulting stage2-merged.zip at the start of
# Stage 3's notebook (it'll prompt you for it automatically).
import shutil

from google.colab import files

shutil.make_archive("/content/stage2-merged", "zip", str(MERGED_OUT_DIR))
files.download("/content/stage2-merged.zip")

shutil.make_archive("/content/stage2-instruction-adapter", "zip", str(ADAPTER_OUT_DIR))
files.download("/content/stage2-instruction-adapter.zip")

# Alternative: persist to Google Drive instead (can hit the same OAuth error above).
# from google.colab import drive
# drive.mount("/content/drive")
# shutil.copytree(ADAPTER_OUT_DIR, "/content/drive/MyDrive/stage2-instruction-adapter", dirs_exist_ok=True)
# shutil.copytree(MERGED_OUT_DIR, "/content/drive/MyDrive/stage2-merged", dirs_exist_ok=True)

## 7. Run the 10 fixed evaluation questions

Same fixed question set as `reports/base_model_evaluation.md` (spec §4/§8), so answers are directly comparable. Copy these into `reports/sft_model_comparison.md` alongside the Stage 1 base-model answers.

In [ ]:
from unsloth import FastLanguageModel

from src.generation_utils import generate
from src.prompts import EVAL_QUESTIONS, SYSTEM_PROMPT

FastLanguageModel.for_inference(model)

for question in EVAL_QUESTIONS:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    answer = generate(model, tokenizer, prompt, max_new_tokens=200, temperature=0.7)
    print(f"Q: {question}\nA: {answer}\n{'-' * 80}")